# Project 09 — BROKEN notebook (debugging exercise)

This notebook fits the **centered** parameterization with a deliberately low `target_accept`, so the funnel produces divergences. Your job: run it, read the divergence count, the **energy plot**, and the **funnel pairs plot**, then apply the non-centered fix. Answer key: `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
y, group, J = data['y'], data['group'], data['J']

### BUG 1 — centered parameterization (the funnel).

Here $\theta_j \sim \text{Normal}(\mu, \tau)$ directly. When $\tau$ is small this couples $\theta_j$ and $\tau$ into a sharp neck.

In [ ]:
with pm.Model(coords={'group': np.arange(J)}) as model:
    mu = pm.Normal('mu', 5.0, 5.0)
    tau = pm.HalfNormal('tau', 2.0)
    sigma = pm.HalfNormal('sigma', 2.0)
    # BUG 1: centered -> Neal's funnel
    theta = pm.Normal('theta', mu=mu, sigma=tau, dims='group')
    pm.Normal('y', mu=theta[group], sigma=sigma, observed=y)
    # BUG 2: target_accept too low for this geometry
    idata = pm.sample(draws=800, tune=1000, chains=4, target_accept=0.8,
                      random_seed=RNG, progressbar=False)

### Symptom — count the divergences and read the diagnostics.

In [ ]:
n_div = int(idata.sample_stats['diverging'].sum())
print('divergences:', n_div)
print(az.summary(idata, var_names=['mu','tau','sigma']))

### Diagnostic 1 — energy plot.

A mismatch between the marginal-energy and energy-transition distributions (low BFMI) is the hierarchical-model fingerprint of the funnel.

In [ ]:
az.plot_energy(idata); plt.tight_layout()

### Diagnostic 2 — the funnel pairs plot.

Plot $\tau$ against a group's $\theta_0$ with divergences highlighted. The red divergent points cluster in the **narrow neck** where $\tau$ is small — the sampler cannot resolve that region.

In [ ]:
az.plot_pair(idata, var_names=['tau','theta'], coords={'group':[0]},
             divergences=True)
plt.tight_layout()

### The fix — non-centered parameterization.

Re-express $\theta_j = \mu + \tau\, z_j$, $z_j \sim \text{Normal}(0,1)$, and raise `target_accept`. The neck disappears and divergences drop to ~0.

In [ ]:
with pm.Model(coords={'group': np.arange(J)}) as model_fixed:
    mu = pm.Normal('mu', 5.0, 5.0)
    tau = pm.HalfNormal('tau', 2.0)
    sigma = pm.HalfNormal('sigma', 2.0)
    z = pm.Normal('z', 0.0, 1.0, dims='group')
    theta = pm.Deterministic('theta', mu + tau * z, dims='group')
    pm.Normal('y', mu=theta[group], sigma=sigma, observed=y)
    idata_fixed = pm.sample(draws=800, tune=1000, chains=4,
                            target_accept=0.95, random_seed=RNG,
                            progressbar=False)
print('divergences after fix:', int(idata_fixed.sample_stats['diverging'].sum()))